# Phase 3：Benchmark 与质量评估

## 今天交付什么？

把“系统很快”“效果不错”改写成可复现的工程结论：在固定数据、固定硬件、固定参数下，查询延迟是多少，top-k 质量是多少，改变一个变量后发生了什么。

**完成后你要能回答：**

1. 为什么第一次查询的时间不能直接当平均延迟？
2. 为什么 P95 比平均值更能暴露长尾？
3. 为什么只报告速度提升是不完整的？
4. 如何区分“没有召回证据”和“召回了但回答不忠实”？

## Evidence Quest 任务卡：Phase 3 总览：速度与质量双榜

**你的身份：** 性能与质量分析师  
**案件背景：** 更快的搜索不一定更有用。你要把速度、召回、排名和参数变化放到同一张实验板上。

### 本关专业 Goal

用可复现 benchmark 选择一个有证据支撑的配置。

### 你要交付的作品

**Benchmark 总览 + 质量-速度结论**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：性能与质量分析师  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 0. 前置知识：Benchmark 是一个公平比较问题

一次可信实验至少要固定：

```text
数据版本 + 模型/索引版本 + 硬件 + 线程数 + 输入长度 + warmup + 迭代次数
```

如果 A 使用短文本、B 使用长文本，或者 A 的第一次加载时间包含在测量里，最后的数字就不能归因于“算法更快”。本 Notebook 先测项目当前真实 BM25 服务，再把优化接口留成可替换边界。

In [1]:
from pathlib import Path
import json
import sys


def find_project_root() -> Path:
    """兼容从项目根目录、notebooks 目录或 JupyterLab 启动目录运行。"""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)

from statistics import mean, median
from time import perf_counter
import platform
import sys

from phase4_mini_rag_system.knowledge_base import KnowledgeBase
from phase2_semantic_search.metrics import recall_at_k, mrr_at_k

knowledge_base = KnowledgeBase()
knowledge_base.ingest(ROOT / "phase1_doc_parser" / "examples" / "input", chunk_size=128, overlap=32)
print("chunks:", len(knowledge_base.chunks))
print("index_version:", knowledge_base.index_version)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship
chunks: 4
index_version: chunks-4-size-128-overlap-32


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase3'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase3
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


## 1. 先理解 warmup：第一次慢不一定是算法慢

第一次调用可能包含 Python 函数首次执行、缓存建立、内存分配或操作系统文件缓存。线上用户可能确实会遇到冷启动，但它和稳定态查询是两个问题，应该分别报告。

In [3]:
def timed_search(query: str, top_k: int = 5) -> float:
    start = perf_counter()
    knowledge_base.search(query, top_k=top_k)
    return (perf_counter() - start) * 1000

first = timed_search("Chunk overlap")
steady = [timed_search("Chunk overlap") for _ in range(20)]
print("first_ms:", round(first, 4))
print("steady_mean_ms:", round(mean(steady), 4))
print("steady_samples_ms:", [round(value, 4) for value in steady[:5]], "...")

first_ms: 0.1631
steady_mean_ms: 0.0309
steady_samples_ms: [0.0779, 0.0406, 0.0332, 0.0264, 0.0337] ...


## 2. P50/P95：为什么不能只看平均值？

把延迟从小到大排序后：

- P50 是中位数，代表一个“典型请求”。
- P95 是 95% 请求不超过的延迟，能看到少数慢请求对体验的影响。
- 平均值会被极端值拉高或拉低，单独使用容易掩盖长尾。

下面使用 nearest-rank 的简单实现。生产报告要写清楚 percentile 定义，避免不同工具的插值规则造成误解。

In [4]:
def percentile(values: list[float], p: float) -> float:
    if not values:
        raise ValueError("values must not be empty")
    ordered = sorted(values)
    index = min(len(ordered) - 1, max(0, int(round(p * len(ordered))) - 1))
    return ordered[index]

def benchmark_search(query: str, *, top_k: int = 5, warmup: int = 5, iterations: int = 50) -> dict[str, float | int]:
    for _ in range(warmup):
        knowledge_base.search(query, top_k=top_k)
    samples = [timed_search(query, top_k) for _ in range(iterations)]
    return {
        "query": query,
        "top_k": top_k,
        "iterations": iterations,
        "mean_ms": round(mean(samples), 4),
        "p50_ms": round(percentile(samples, 0.50), 4),
        "p95_ms": round(percentile(samples, 0.95), 4),
    }

benchmark = benchmark_search("Chunk overlap")
print(benchmark)

{'query': 'Chunk overlap', 'top_k': 5, 'iterations': 50, 'mean_ms': 0.055, 'p50_ms': 0.0501, 'p95_ms': 0.0817}


**读数纪律：** 当前语料只有几个 Chunk，所以这个延迟不能代表生产规模。它的价值在于建立 harness：以后替换 Dense、Faiss、ONNX 或更大数据集时，仍然用同一套测量方法，比较才公平。

In [5]:
benchmarks = [
    benchmark_search("Chunk overlap", top_k=k)
    for k in (1, 2, 5)
]
for row in benchmarks:
    print(row)
print("观察：top_k 同时影响返回结果量和潜在质量；结论需要和 qrels 一起看。")

{'query': 'Chunk overlap', 'top_k': 1, 'iterations': 50, 'mean_ms': 0.0282, 'p50_ms': 0.0253, 'p95_ms': 0.0422}
{'query': 'Chunk overlap', 'top_k': 2, 'iterations': 50, 'mean_ms': 0.0324, 'p50_ms': 0.0252, 'p95_ms': 0.056}
{'query': 'Chunk overlap', 'top_k': 5, 'iterations': 50, 'mean_ms': 0.0356, 'p50_ms': 0.0335, 'p95_ms': 0.0594}
观察：top_k 同时影响返回结果量和潜在质量；结论需要和 qrels 一起看。


## 3. 质量闸门：快但错，仍然是失败

把 Phase 2 的 qrels 带过来。对于每个配置同时记录：

- `Recall@k`：有没有把相关 Chunk 召回？
- `MRR@k`：第一个相关 Chunk 排名是否靠前？
- `P95 latency`：长尾请求有多慢？

这三个数字共同描述“用户能否及时看到正确证据”。

In [6]:
chunks = knowledge_base.chunks
def first_id_containing(text: str) -> str:
    for item in chunks:
        if text.lower() in str(item["text"]).lower():
            return str(item["id"])
    return str(chunks[0]["id"])

evaluation_queries = {
    "q-001": {"text": "Chunk overlap", "relevant": {first_id_containing("overlap")}},
    "q-002": {"text": "Dense BM25", "relevant": {first_id_containing("Dense")}},
}

for query_id, case in evaluation_queries.items():
    result_ids = [item["chunk_id"] for item in knowledge_base.search(case["text"], top_k=5)]
    print(query_id, {
        "ranked_ids": result_ids,
        "recall@5": recall_at_k(result_ids, case["relevant"], k=5),
        "mrr@5": mrr_at_k(result_ids, case["relevant"], k=5),
    })

q-001 {'ranked_ids': ['82ddc7d612e87b02', '3692b05e025373a8'], 'recall@5': 1.0, 'mrr@5': 1.0}
q-002 {'ranked_ids': ['5a62fff245b307a5', '83510b23d2680327'], 'recall@5': 1.0, 'mrr@5': 0.5}


## 4. 单变量实验：Chunk 参数是否会改变系统？

现在只改变 `chunk_size/overlap`，查询、输入文件和评估问题不变。我们同时记录 Chunk 数、Recall 和 P95。这样才能回答“更细的 Chunk 是否值得它带来的索引成本”，而不是凭经验争论 256 还是 512。

In [7]:
config_results = []
for chunk_size, overlap in ((64, 16), (128, 32), (256, 64)):
    kb = KnowledgeBase()
    kb.ingest(ROOT / "phase1_doc_parser" / "examples" / "input", chunk_size=chunk_size, overlap=overlap)
    start = perf_counter()
    ids = [item["chunk_id"] for item in kb.search("Chunk overlap", top_k=5)]
    elapsed = (perf_counter() - start) * 1000
    relevant = {str(item["id"]) for item in kb.chunks if "overlap" in str(item["text"]).lower()}
    config_results.append({
        "chunk_size": chunk_size,
        "overlap": overlap,
        "chunks": len(kb.chunks),
        "one_query_ms": round(elapsed, 4),
        "recall@5": recall_at_k(ids, relevant, k=5),
    })
for row in config_results:
    print(row)

{'chunk_size': 64, 'overlap': 16, 'chunks': 6, 'one_query_ms': 0.0647, 'recall@5': 1.0}
{'chunk_size': 128, 'overlap': 32, 'chunks': 4, 'one_query_ms': 0.0617, 'recall@5': 1.0}
{'chunk_size': 256, 'overlap': 64, 'chunks': 2, 'one_query_ms': 0.0688, 'recall@5': 1.0}


### 如何避免过度解读？

这个实验只有一个查询，不能宣布某个参数“最佳”。它只能告诉你：参数确实同时影响数据规模和结果，需要扩大 qrels 后再做结论。专业报告会写：

> 在当前数据版本和一个示例 Query 上观察到……；该现象尚不足以证明普遍规律，下一步扩充标注集并重复实验。

谨慎不是保守，而是让简历和技术报告里的数字经得起追问。

## 5. 错误归因：评估分数低到底是谁的问题？

把一次失败拆成链路，而不是只给系统打一个总分：

```text
文件解析失败 -> 没有正确 Chunk
正确 Chunk 不在 top-k -> 召回/分块问题
正确 Chunk 在 top-k，但答案缺事实 -> 上下文编排/生成问题
答案包含证据没有支持的内容 -> 忠实性问题
评估器与人工判断冲突 -> 评估集或 judge 问题
```

下面用一条小表把“现象”映射到下一步只改变的变量。

In [8]:
failure_cases = [
    {"observed": "相关 Chunk 不在 top-k", "layer": "retrieval", "next_change": "只改 tokenizer 或检索策略"},
    {"observed": "相关 Chunk 在 top-k，回答缺少事实", "layer": "generation/context", "next_change": "只改上下文截断或 prompt"},
    {"observed": "答案添加了证据中没有的数字", "layer": "faithfulness", "next_change": "增加引用约束并做人工复核"},
]
for case in failure_cases:
    print(case)

{'observed': '相关 Chunk 不在 top-k', 'layer': 'retrieval', 'next_change': '只改 tokenizer 或检索策略'}
{'observed': '相关 Chunk 在 top-k，回答缺少事实', 'layer': 'generation/context', 'next_change': '只改上下文截断或 prompt'}
{'observed': '答案添加了证据中没有的数字', 'layer': 'faithfulness', 'next_change': '增加引用约束并做人工复核'}


## 6. ONNX/INT8 为什么先做“准备检查”？

量化不是自动加速按钮。它可能带来：模型更小、推理更快，也可能带来输出偏差、Recall 下降或 P95 变差。真实实验必须同时保存：

```text
模型/索引版本、文件大小、mean/P50/P95、cosine 相似度、Recall@k、设备和线程数
```

当前 Notebook 不伪造量化结果，只检查可选依赖和记录当前环境。等模型导出完成后，沿用本 Notebook 的 benchmark harness。

In [9]:
import importlib.util

print({
    "onnx": importlib.util.find_spec("onnx") is not None,
    "onnxruntime": importlib.util.find_spec("onnxruntime") is not None,
    "psutil": importlib.util.find_spec("psutil") is not None,
})
print("当前 benchmark 环境:")
print({"python": sys.version.split()[0], "platform": platform.platform(), "processor": platform.processor() or "unknown"})

{'onnx': False, 'onnxruntime': True, 'psutil': True}
当前 benchmark 环境:
{'python': '3.13.5', 'platform': 'Windows-11-10.0.22631-SP0', 'processor': 'AMD64 Family 25 Model 68 Stepping 1, AuthenticAMD'}


## Phase 3 阶段闸门

- [ ] 解释 warmup、mean、P50、P95 各自回答什么问题。
- [ ] 延迟数据包含输入规模、迭代次数、设备和版本信息。
- [ ] 同一实验同时报告速度和质量，不把一次测量当结论。
- [ ] 能把失败归因到解析、召回、上下文、生成或评估层。
- [ ] 能写出量化实验的质量闸门，而不是只追求更小模型。

**项目交付物：** baseline benchmark 表、参数对照结果、错误归因记录。下一阶段把这条经过验证的链路包装成别人可以调用的服务。

## Boss Challenge：把一个参数变化同时放到质量表和延迟表，写出有边界的工程结论。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [10]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [11]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['docs/phase3_baseline_report.md', 'data/processed/phase3_timing_baseline.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['docs\\phase3_baseline_report.md', 'data\\processed\\phase3_timing_baseline.json']
